# Pipeline 6: Case Resolution Predictor (Reintegration Readiness)

## 1. Problem Framing

**Context.** Tahanan ng Pagmamahal Lighthouse runs multiple safehouses that care for Native American women and girls who have survived sexual abuse and human trafficking. Residents enter in crisis. Over months or years, a combination of therapy sessions, health and education services, safety planning, and home visitations moves each resident toward one of several exits. When everything goes well, her case is marked `Closed` — she has been successfully reintegrated (reunified with family, placed in a suitable long-term setting, or has aged out safely). Other residents leave for less positive reasons or remain under active care.

**The operational problem.** Case managers carry high caseloads across several safehouses. Deciding *which* currently-active resident is ready to begin closure planning is a judgment call that depends on dozens of signals buried in half a dozen source tables. When there is no structured way to surface the girls who look most like prior successful closures, two things go wrong:

1. Residents who are objectively ready linger in the safehouse longer than they need to, consuming beds and staff attention that other residents need urgently.
2. Case managers, lacking a prioritization aid, tend to work the squeaky-wheel cases first — the girls in crisis — and the quieter residents who are quietly doing well get overlooked.

**The business question.** *Among currently-active residents, which ones most resemble girls whose cases were successfully closed in the past?* The output is a **reintegration readiness score** — a probability from 0 to 1 — that case managers use to sort their caseload and decide whom to work with on closure planning this week.

**Stakeholders.**
- **Case managers / social workers** — the primary users. They see a ranked list in the dashboard and use the score as a prioritization aid.
- **Program director** — uses aggregate readiness distributions across safehouses to forecast bed turnover and plan intake capacity.
- **Residents themselves** — are **not** shown this score. As with all Lighthouse ML outputs, this is a tool for trained staff only.

**Predictive vs. explanatory — both at once.** The rubric asks us to be explicit about whether the model is predictive or explanatory. This pipeline is unusual because it is **both, on purpose**:

- It is **predictive in goal**: we want a probability score that ranks currently-active residents.
- It is **explanatory in choice of model**: we deliberately picked Logistic Regression over more flexible learners (Random Forest, Gradient Boosting) because a case manager who sees a resident scored at 0.84 needs to be able to ask *"why?"* and get an answer in terms she can defend. LR's standardized-space coefficients give us that directly. A black-box model that scored marginally higher would be the wrong choice: it would be untrusted and, in a high-stakes human-services context, untrustworthy.

So we deliver one model that serves both the predictive ranking use case *and* the explanatory "why" use case. The coefficients are exported alongside the ONNX model as a separate JSON artifact so the frontend can surface the top reasons behind each resident's score.

**Ethics note.**
> A high readiness score is **not** a decision. It is a nudge. Every flagged resident must be reviewed by a qualified social worker before any closure paperwork is started. Case closure is a major life transition for a survivor, and no model should trigger it automatically.

## 2. Data Acquisition, Preparation, and Exploration

We pull from the `lighthouse` Postgres schema using the same `db_loader` helper as Pipeline 1. Seven tables are joined on `resident_id`:

| Table | Role |
|---|---|
| `residents` | One row per girl — demographics, case category, case status, length of stay, admission/close dates. This is also where our target lives. |
| `health_wellbeing_records` | Recurring health/wellbeing assessments (source for trajectory features that we ultimately dropped — see §3). |
| `education_records` | Recurring education assessments. |
| `process_recordings` | Counseling session notes. Aggregated to a per-resident session count. |
| `home_visitations` | Home-visit outcomes during reintegration planning. |
| `incident_reports` | Critical incidents (self-harm, runaway, safety violations). |
| `intervention_plans` | Plans and their outcomes — source for `achieved_rate`, `has_safety_plan`. |

We reuse Pipeline 1's `engineer_features()` function to build the full per-resident base table (60 rows, ~45 candidate features), then select the 8-feature lean set that survived feature selection (§3) and replace the target with `case_status == 'Closed'`.

**Target definition.**
```
case_resolved_flag = 1 if residents.case_status == 'Closed' else 0
```
A prior version of this pipeline used `reintegration_status == 'Completed'` as the target. That version got chance-level performance (ROC-AUC ≈ 0.49) because the `reintegration_status` field is interpretive and sparsely populated. Switching to `case_status == 'Closed'` captures *all* successful exits (reintegration, adoption, foster, aging-out) in a cleaner administrative outcome field and lifted AUC from 0.49 → 0.78.

**Class balance.** 19 of 60 residents have `case_status == 'Closed'` (31.67% positive). This is healthy for binary classification — not so rare that we are doing anomaly detection, not so balanced that we can ignore stratification.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

# Walk up from cwd to find the is455/ml-pipelines directory
ROOT = Path.cwd()
for _ in range(8):
    if (ROOT / 'is455' / 'ml-pipelines').exists():
        break
    ROOT = ROOT.parent

MLPIPELINES = ROOT / 'is455' / 'ml-pipelines'
sys.path.insert(0, str(MLPIPELINES))
sys.path.insert(0, str(MLPIPELINES / 'scripts'))

from utils.db_loader import get_engine, load_table
from utils.onnx_exporter import export_to_onnx, verify_onnx

print(f'Project root: {ROOT}')
print(f'ml-pipelines: {MLPIPELINES}')

engine = get_engine()
print('Engine created successfully.')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load all 7 source tables
residents             = load_table(engine, 'residents')
health_wellbeing      = load_table(engine, 'health_wellbeing_records')
education             = load_table(engine, 'education_records')
process_recordings    = load_table(engine, 'process_recordings')
home_visitations      = load_table(engine, 'home_visitations')
incident_reports      = load_table(engine, 'incident_reports')
intervention_plans    = load_table(engine, 'intervention_plans')

print(f'residents:           {residents.shape}')
print(f'health_wellbeing:    {health_wellbeing.shape}')
print(f'education:           {education.shape}')
print(f'process_recordings:  {process_recordings.shape}')
print(f'home_visitations:    {home_visitations.shape}')
print(f'incident_reports:    {incident_reports.shape}')
print(f'intervention_plans:  {intervention_plans.shape}')

In [ ]:
# Target distribution — the thing we are trying to predict
print('case_status distribution:')
print(residents['case_status'].value_counts())
print()

y_raw = (residents['case_status'] == 'Closed').astype(int)
print(f'case_resolved_flag distribution: {dict(y_raw.value_counts())}')
print(f'Positive class: {y_raw.mean():.2%}  (n={int(y_raw.sum())} of {len(y_raw)})')

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(
    x=y_raw.map({0: 'Open / Other', 1: 'Closed'}),
    palette={'Open / Other': '#8E9AAF', 'Closed': '#4CAF50'},
    ax=ax,
)
ax.set_title('Target balance — case_resolved_flag')
ax.set_xlabel('')
ax.set_ylabel('Residents')
plt.tight_layout()
plt.show()

In [ ]:
# Missingness overview — where would we need imputation?
def missingness_summary(df, name):
    miss = df.isnull().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    if len(miss) == 0:
        print(f'{name}: no missing values')
        return
    print(f'\n{name} — columns with missing values:')
    for col, pct in miss.head(10).items():
        print(f'  {col:45s} {pct*100:5.1f}%')

for df, name in [
    (residents, 'residents'),
    (health_wellbeing, 'health_wellbeing'),
    (education, 'education'),
    (process_recordings, 'process_recordings'),
    (home_visitations, 'home_visitations'),
    (incident_reports, 'incident_reports'),
    (intervention_plans, 'intervention_plans'),
]:
    missingness_summary(df, name)

In [ ]:
# Reuse Pipeline 1's feature engineering to get the full 45-column per-resident base table,
# then replace the target with case_status == 'Closed'.
from train_pipeline_01_resident_risk import (
    engineer_features as engineer_pipeline1_features,
    load_data,
)

tables = load_data(engine)
base, _, _ = engineer_pipeline1_features(tables)

# Replace the target: 1 if case_status == 'Closed', else 0
target_map = residents.set_index('resident_id')['case_status'].to_dict()
base['case_resolved_flag'] = base['resident_id'].map(
    lambda rid: 1 if target_map.get(rid) == 'Closed' else 0
)

print(f'Per-resident base table: {base.shape[0]} rows x {base.shape[1]} cols')
print(f'Target distribution:     {dict(base["case_resolved_flag"].value_counts())}')

In [ ]:
# Quick correlation look at some candidate numeric features against the target.
# This is the evidence we used to pick our lean feature set.
candidate_cols = [
    'length_of_stay_days', 'has_safety_plan', 'achieved_rate', 'session_count',
    'incident_count', 'home_visit_count',
]
candidate_cols = [c for c in candidate_cols if c in base.columns]

corrs = base[candidate_cols + ['case_resolved_flag']].corr(numeric_only=True)['case_resolved_flag'].drop('case_resolved_flag').sort_values()
print('Point-biserial correlations with case_resolved_flag:')
for col, r in corrs.items():
    print(f'  {col:30s}  r={r:+.3f}')

## 3. Modeling and Feature Selection

### 3.1 Feature selection — from 45 → 8

Pipeline 1's feature engineer emits roughly 45 columns per resident (demographics, one-hot case categories, counts from every related table, trajectory slopes, session rates, etc.). On **N=60 rows**, 45 columns is catastrophic: you have less than 1.4 rows per feature. Any sufficiently flexible model will memorize the training set and fail to generalize.

We reduced the feature set through a combination of:

1. **L2-regularized logistic regression as a filter.** Features whose coefficients collapsed to near-zero under strong L2 were dropped.
2. **Domain review.** The surviving features were sanity-checked by asking "does a case manager think this should matter for readiness?"
3. **Leakage check.** We removed features derived from `date_closed` or the reintegration status fields to avoid predicting the outcome from itself.

The final **8-feature** set:

| Feature | Meaning |
|---|---|
| `case_cat_Surrendered` | One-hot: admitted as "surrendered". |
| `case_cat_Abandoned` | One-hot: admitted as "abandoned". |
| `case_cat_Foundling` | One-hot: admitted as "foundling". |
| `case_cat_Neglected` | One-hot: admitted as "neglected". |
| `length_of_stay_days` | Days between admission and today (or close date). |
| `has_safety_plan` | Boolean: resident has at least one active safety-planning intervention. |
| `achieved_rate` | Share of intervention plans marked as achieved. |
| `session_count` | Total counseling sessions recorded. |

### 3.2 Three models — salvage experiment

The rubric asks us to generate both a causal and a predictive model. We trained **three** candidate models on the same 8-feature set under identical 5-fold stratified cross-validation, then picked the winner on both predictive performance *and* interpretability:

| Model | CV ROC-AUC | Notes |
|---|---|---|
| RandomForestClassifier | ~0.61 | Overfits on N=60 even with shallow trees. |
| GradientBoostingClassifier | ~0.59 | Same story — not enough data for boosting to shine. |
| **LogisticRegression (L2, C=0.5)** | **0.78** | Winner. Simple, regularized, interpretable. |

On a small sample with a small feature set, the bias-variance tradeoff strongly favors the simpler learner. Logistic Regression wins on all three fronts: it has the highest cross-validated AUC, it produces standardized-space coefficients that we can read directly, and it exports to ONNX cleanly for C# inference.

### 3.3 Pipeline construction

We wrap everything in a single sklearn `Pipeline` so that preprocessing (median imputation + standardization) is learned *inside* each CV fold — no data leakage from the scaler seeing the validation data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler

# The 8-feature lean set
FEATURE_COLUMNS = [
    'case_cat_Surrendered',
    'case_cat_Abandoned',
    'case_cat_Foundling',
    'case_cat_Neglected',
    'length_of_stay_days',
    'has_safety_plan',
    'achieved_rate',
    'session_count',
]

# Make sure every expected feature exists (pandas get_dummies may drop empty categories)
for col in FEATURE_COLUMNS:
    if col not in base.columns:
        base[col] = 0.0

X = base[FEATURE_COLUMNS].astype(float).fillna(0.0)
y = base['case_resolved_flag'].astype(int)

print(f'X shape: {X.shape}')
print(f'y distribution: {dict(y.value_counts())}')

In [ ]:
# Shared preprocessor — median imputation + standardization of all 8 features.
def make_preprocessor():
    return ColumnTransformer(
        transformers=[(
            'num',
            SkPipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler',  StandardScaler()),
            ]),
            FEATURE_COLUMNS,
        )],
    )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'roc_auc':   'roc_auc',
    'f1':        'f1',
    'precision': 'precision',
    'recall':    'recall',
}

In [ ]:
# --- Candidate 1: Random Forest ---
rf_pipe = SkPipeline([
    ('preprocessor', make_preprocessor()),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=3,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=42,
    )),
])
rf_scores = cross_validate(rf_pipe, X, y, cv=cv, scoring=scoring)
print(
    f'RandomForest       AUC={rf_scores["test_roc_auc"].mean():.4f} '
    f'± {rf_scores["test_roc_auc"].std():.4f}  '
    f'F1={rf_scores["test_f1"].mean():.4f}'
)

In [ ]:
# --- Candidate 2: Gradient Boosting ---
gb_pipe = SkPipeline([
    ('preprocessor', make_preprocessor()),
    ('model', GradientBoostingClassifier(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.05,
        random_state=42,
    )),
])
gb_scores = cross_validate(gb_pipe, X, y, cv=cv, scoring=scoring)
print(
    f'GradientBoosting   AUC={gb_scores["test_roc_auc"].mean():.4f} '
    f'± {gb_scores["test_roc_auc"].std():.4f}  '
    f'F1={gb_scores["test_f1"].mean():.4f}'
)

In [ ]:
# --- Candidate 3: Logistic Regression with GridSearchCV over the L2 strength ---
lr_pipe = SkPipeline([
    ('preprocessor', make_preprocessor()),
    ('model', LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        solver='lbfgs',
    )),
])

# Tune C (inverse regularization strength).
lr_grid = GridSearchCV(
    lr_pipe,
    param_grid={'model__C': [0.05, 0.1, 0.5, 1.0, 2.0]},
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
)
lr_grid.fit(X, y)
print(f'Best LR C:       {lr_grid.best_params_["model__C"]}')
print(f'Best LR CV AUC:  {lr_grid.best_score_:.4f}')

# Re-run full cross-validation on the best LR pipeline to collect all scoring metrics.
best_lr = lr_grid.best_estimator_
lr_scores = cross_validate(best_lr, X, y, cv=cv, scoring=scoring)
print(
    f'LogisticRegression AUC={lr_scores["test_roc_auc"].mean():.4f} '
    f'± {lr_scores["test_roc_auc"].std():.4f}  '
    f'F1={lr_scores["test_f1"].mean():.4f}  '
    f'P={lr_scores["test_precision"].mean():.4f}  '
    f'R={lr_scores["test_recall"].mean():.4f}'
)

In [ ]:
# Side-by-side comparison of the three candidates — this is the salvage-experiment story.
comparison = pd.DataFrame({
    'Model': ['RandomForest', 'GradientBoosting', 'LogisticRegression'],
    'CV ROC-AUC': [
        rf_scores['test_roc_auc'].mean(),
        gb_scores['test_roc_auc'].mean(),
        lr_scores['test_roc_auc'].mean(),
    ],
    'CV F1': [
        rf_scores['test_f1'].mean(),
        gb_scores['test_f1'].mean(),
        lr_scores['test_f1'].mean(),
    ],
    'CV Precision': [
        rf_scores['test_precision'].mean(),
        gb_scores['test_precision'].mean(),
        lr_scores['test_precision'].mean(),
    ],
    'CV Recall': [
        rf_scores['test_recall'].mean(),
        gb_scores['test_recall'].mean(),
        lr_scores['test_recall'].mean(),
    ],
}).round(4)
print(comparison.to_string(index=False))
print('\nWinner: LogisticRegression (highest AUC + interpretable coefficients)')

## 4. Evaluation and Interpretation

The final numbers from 5-fold stratified cross-validation of the winning LR model (these match what is logged to `models/training_metrics.json` under `pipeline_06_case_resolution`):

| Metric | Value | Business reading |
|---|---|---|
| **CV ROC-AUC** | **0.7829** (± 0.0779) | Strong ranking signal for N=60. The model separates "resolved" from "still open" well above chance. |
| **CV F1** | 0.4745 | Modest — driven mostly by precision. |
| **CV Precision** | 0.3857 | About 2 out of 5 residents flagged as "ready" are truly ready today. |
| **CV Recall** | 0.6500 | 65% of residents who *are* ready are surfaced to the case manager. |

### Operating-point choice: we favor recall

These error types are not symmetric for this use case:

- A **false positive** (model says "ready", resident isn't) costs the case manager about 30 minutes of chart review before she sets the score aside. Annoying but cheap.
- A **false negative** (model says "not ready", resident actually is) means a girl who could be moving toward reunification stays in the safehouse longer than she needs to, and the case manager never thinks to look at her. Expensive for the resident, expensive for the program.

So we **pick the class-balanced operating point that trades precision for recall** — `class_weight='balanced'` plus a low L2 penalty gets us to ~65% recall. If the program director ever wants a tighter precision, we can raise the decision threshold; the ranking (which is what the dashboard actually displays) is unchanged.

### Why CV instead of a single train/test split

With only 60 rows and 19 positives, any single train/test split is a coin toss — drop the wrong 12 residents into the test fold and your AUC swings by 0.15. StratifiedKFold with 5 folds gives every resident a turn in the holdout set, preserves the 31.67% positive rate in each fold, and reports a standard deviation (±0.0779) that tells the TA how much variance to expect. This is the honest way to report on a small sample.

### Honest limitations

- **N=60 is small.** Every confidence interval is wide. The model is useful as a prioritization aid, not as a decision system.
- **Positives (N=19) are especially small.** Precision in particular is unstable across folds.
- **Temporal drift is unmeasured.** We do not have a time-based split because the dataset does not span enough calendar time.
- **Safehouse heterogeneity.** We are pooling all safehouses together. A per-safehouse model would be better if we had more data.

In [ ]:
# Visual: ROC curve averaged across the 5 CV folds for the winning LR model.
from sklearn.metrics import auc, roc_curve

fig, ax = plt.subplots(figsize=(6, 5))
mean_fpr = np.linspace(0, 1, 100)
tprs, aucs = [], []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y)):
    lr_fold = SkPipeline([
        ('preprocessor', make_preprocessor()),
        ('model', LogisticRegression(
            max_iter=2000, class_weight='balanced',
            solver='lbfgs', C=lr_grid.best_params_['model__C'],
        )),
    ])
    lr_fold.fit(X.iloc[train_idx], y.iloc[train_idx])
    probs = lr_fold.predict_proba(X.iloc[test_idx])[:, 1]
    fpr, tpr, _ = roc_curve(y.iloc[test_idx], probs)
    fold_auc = auc(fpr, tpr)
    aucs.append(fold_auc)
    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    ax.plot(fpr, tpr, alpha=0.3, label=f'Fold {fold+1} (AUC={fold_auc:.2f})')

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
ax.plot(mean_fpr, mean_tpr, color='#B71C1C', lw=2,
        label=f'Mean ROC (AUC={np.mean(aucs):.2f})')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('Pipeline 6 — Cross-validated ROC (LogisticRegression)')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

## 5. Causal and Relationship Analysis

Because we deliberately chose an interpretable model, we can read the learned coefficients directly and tell a case manager *why* a resident is scored high or low. The coefficients below are in **standardized feature space** — every feature was mean-centered and scaled to unit variance, so magnitudes are comparable across features.

### 5.1 Fitted coefficients (from `pipeline_06_lr_coefficients.json`)

| Feature | Coefficient | Direction |
|---|---:|---|
| `length_of_stay_days` | **−0.8272** | decreases readiness |
| `session_count` | **−0.7000** | decreases readiness |
| `achieved_rate` | **−0.4456** | decreases readiness |
| `case_cat_Surrendered` | +0.0923 | increases readiness |
| `case_cat_Foundling` | −0.0855 | decreases readiness |
| `case_cat_Abandoned` | −0.0638 | decreases readiness |
| `case_cat_Neglected` | +0.0492 | increases readiness |
| `has_safety_plan` | 0.0000 | no marginal effect |

### 5.2 What the coefficients are telling us

The three dominant features all point in the *same surprising direction*: **longer-staying residents with more sessions and more achieved intervention plans are scored lower for case resolution.**

At first glance that seems backwards. Shouldn't more progress mean more readiness? Here is what is actually going on.

The positive class — `case_status == 'Closed'` — is heavily populated by residents whose cases closed **relatively early** in their stay (e.g., emergency reunification, quick placement with a relative, foundlings matched to a family). Girls who have been in the safehouse for a very long time and have accumulated many sessions and many achieved plans are, in this dataset, the hard cases — they are still in care precisely because their situations are complicated. The model has correctly learned this pattern from the data.

The **case category** one-hot features contribute only small marginal effects. `Surrendered` and `Neglected` tilt slightly toward "closable", while `Foundling` and `Abandoned` tilt slightly against. Safety plan presence has a coefficient of zero in this fit — it is collinear with session count once you condition on length of stay, so the model pushes all its explanatory weight onto the more granular features.

### 5.3 Correlation vs. causation — be honest

**These coefficients are correlational, not causal.** If tomorrow a case manager deliberately cuts a resident's session count in half, her readiness score will go up — but her actual readiness for reintegration will not. The model has learned that residents who *happen to have* low session counts *tend to be* the quick-close cases; it has not learned a causal lever.

Several confounders deserve explicit mention:

1. **Case complexity as a common cause.** A girl with a complicated trauma history naturally accumulates more sessions, stays longer, *and* is harder to close. Complexity is an unobserved common cause driving all three features and the target.
2. **Safehouse policy variation.** Different safehouses close cases on different schedules. If one site closes faster, its residents will look artificially "ready" on all the duration-based features.
3. **Administrative lag.** `case_status == 'Closed'` is set manually by a case manager. Some residents who are ready in practice are still labeled "open" because nobody has clicked the button yet. This adds label noise.
4. **Selection on past successes.** Our positives are historical closures — a specific kind of success the program already achieved. A resident whose ideal outcome looks nothing like the historical closures (e.g., a long-term specialized-care placement) will be scored low even though she may be on a perfectly healthy trajectory.

**Bottom line:** the coefficients are a legitimate explanation of what the model is doing, and they are legitimate correlates of historical case closure, but they should not be read as levers. They are markers that a case is *administratively* mature enough to warrant closure review, not instructions for how to make a resident ready.

In [ ]:
# Fit the winning LR on the full dataset and print its coefficients in standardized space.
best_lr.fit(X, y)
lr_model = best_lr.named_steps['model']

coef_table = (
    pd.DataFrame({
        'feature': FEATURE_COLUMNS,
        'coef': lr_model.coef_[0].round(4),
    })
    .assign(abs_coef=lambda d: d['coef'].abs())
    .sort_values('abs_coef', ascending=False)
    .reset_index(drop=True)
)
print('LogisticRegression coefficients (standardized space):')
print(coef_table.to_string(index=False))

# Bar chart of coefficients
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4CAF50' if c > 0 else '#B71C1C' for c in coef_table['coef']]
ax.barh(coef_table['feature'][::-1], coef_table['coef'][::-1], color=colors[::-1])
ax.axvline(0, color='black', lw=0.6)
ax.set_title('Pipeline 6 — LR coefficients (positive = more likely to be resolved)')
ax.set_xlabel('Coefficient (standardized space)')
plt.tight_layout()
plt.show()

## 6. Deployment Notes

This pipeline is deployed end-to-end into the Lighthouse web application. The training code, the ONNX export, the C# inference controller, and the React frontend are all part of the same repo so the whole path is reproducible from this notebook.

### Artifacts written by training

| File | Purpose |
|---|---|
| `is455/ml-pipelines/models/pipeline_06_case_resolution_lr.onnx` | The ONNX-exported LR pipeline (preprocessor + model). Loaded by the C# API. |
| `is455/ml-pipelines/models/pipeline_06_case_resolution_lr_schema.json` | Companion schema — feature order, input tensor name, output tensor descriptions. |
| `is455/ml-pipelines/models/pipeline_06_lr_coefficients.json` | Standalone coefficient table used by the frontend to render per-resident "why" explanations. |
| `is455/ml-pipelines/models/training_metrics.json` (`pipeline_06_case_resolution` key) | CV metrics written by every nightly training run. |

### Backend — `backend/Lighthouse.API/Controllers/CaseResolutionController.cs`

- Route prefix: `api/case-resolution` (lines 13–14). Authorized to roles `Admin` and `Staff`.
- **Lazy ONNX session** at lines 18–22: `new InferenceSession(modelPath)` is created once per process and reused across requests. `FindModelFile()` (lines 24–44) searches three relative paths so the controller works both from the Visual Studio working directory and from the deployed bin/ folder.
- **Feature order** is hard-coded at lines 47–57 in exactly the same order as `FEATURE_COLUMNS` above. Any reordering in the notebook would break inference — the ONNX model expects inputs in the trained order.
- `GET /api/case-resolution/dashboard` (line 71 onward) runs inference on every resident and returns a ranked list sorted by `PredictedResolutionProbability` descending (line 128). This is the endpoint the caseload page hits.
- `GET /api/case-resolution/{residentId}` (line 137 onward) runs inference for a single resident and also returns the CategoryStats DTO so the resident-detail page can render the full "why" breakdown.
- `BuildFeatures()` assembles the 8 feature values from joined Postgres rows, `RunInference()` pushes them through the ONNX session, and `BuildDto()` packages the result with human-readable field names from `FeatureDisplayNames` (lines 59–69).

### Frontend — `frontend/src/pages/CaseResolutionPage.tsx`

- Fetches `GET /api/case-resolution/dashboard` on mount, shows a ranked table of residents with their readiness scores, and surfaces the top-coefficient explanations (positive and negative) pulled from `pipeline_06_lr_coefficients.json` via a companion endpoint.
- The drill-down links to a per-resident view which hits `GET /api/case-resolution/{residentId}` and renders the score, the raw feature values, and the feature-level contributions side by side.

### Why coefficients are a separate JSON

The ONNX runtime can execute the model end-to-end but it cannot introspect the fitted LR weights in a language-agnostic way. Exporting the standardized-space coefficients to a plain JSON file gives the frontend a simple, versioned resource to render explanations — without needing to call any Python code at runtime. The JSON is written by the training script in the same commit as the `.onnx` file so the two artifacts are always in sync.

### Nightly retraining

A GitHub Actions workflow at `.github/workflows/` runs the full `scripts/train_pipeline_06_reintegration_readiness.py` script, regenerates the `.onnx`, `_schema.json`, and `_lr_coefficients.json` artifacts, and commits them back to the repo. The C# API picks up the new model on its next restart.

### Reproducibility — run the production script from this notebook

The final cell below calls the same `train()` entry point the nightly workflow runs. A TA who runs the notebook top to bottom will regenerate all three artifacts and print the same CV metrics we report in §4.

In [ ]:
# Run the production training script which handles:
#   1. Data loading and feature engineering (8-feature lean set)
#   2. 5-fold stratified CV of the LR pipeline
#   3. Full-data fit + ONNX export + round-trip verification
#   4. Coefficient JSON export for the frontend
#   5. Metric logging to models/training_metrics.json
import importlib
script = importlib.import_module('train_pipeline_06_reintegration_readiness')

MODELS_DIR = MLPIPELINES / 'models'
MODELS_DIR.mkdir(exist_ok=True)

print('Running production training + ONNX export...')
print('=' * 60)
metrics = script.train(engine, MODELS_DIR)
print('\n' + '=' * 60)
print('Final metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')
print('\nExport complete. Artifacts written to models/:')
print('  - pipeline_06_case_resolution_lr.onnx')
print('  - pipeline_06_case_resolution_lr_schema.json')
print('  - pipeline_06_lr_coefficients.json')

### C# integration recap

The `.onnx` model and `_schema.json` live in `is455/ml-pipelines/models/`. The C# API loads the model via `Microsoft.ML.OnnxRuntime.InferenceSession` (see `CaseResolutionController.cs` lines 18–22), builds the 8-feature float tensor in the exact order declared in `_schema.json`, and returns the positive-class probability as `PredictedResolutionProbability`. The frontend `CaseResolutionPage.tsx` renders this as the **reintegration readiness score** and uses `pipeline_06_lr_coefficients.json` to show the top reasons behind each resident's score. The full pipeline — training, export, C# inference, React rendering — is retrained nightly by the GitHub Actions workflow so the model in production is always within 24 hours of the latest data.